## **Car Prediction Project | Predicting**
1. User input Schema
2. Input Handling => replace_categorical_by_numerical... Engine Volume | Levy | Mileage
3. Numerical Feature Engineering => adding age
4. Dropping => "ID", "Doors", "Prod. year", "Cylinders"
5. Categorical Feature Engineering => One-Hot Encoded , Label Encoded
6. Numerical Scaling 

In [220]:
import pickle
import pandas as pd
from datetime import datetime

In [221]:
car_dict = {
    "Levy": 831,
    "Manufacturer": "HYUNDAI",
    "Model": "Sonata",
    "Prod. year": 2011,
    "Category": "Sedan",
    "Leather interior": "Yes",
    "Fuel type": "Petrol",
    "Engine volume": 2.4,
    "Mileage": 161600,
    "Gear box type": "Tiptronic",
    "Drive wheels": "Front",
    "Wheel": "Left wheel",
    "Color": "Red",
    "Airbags": 8
}

In [222]:
data = pd.DataFrame([car_dict])
data

,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Gear box type,Drive wheels,Wheel,Color,Airbags
0,831,HYUNDAI,Sonata,2011,Sedan,Yes,Petrol,2.4,161600,Tiptronic,Front,Left wheel,Red,8


In [223]:
data.dtypes

Levy                  int64
Manufacturer            str
Model                   str
Prod. year            int64
Category                str
Leather interior        str
Fuel type               str
Engine volume       float64
Mileage               int64
Gear box type           str
Drive wheels            str
Wheel                   str
Color                   str
Airbags               int64
dtype: object

In [224]:
data['Age'] = datetime.now().year - data['Prod. year']
data = data.drop(columns=['Prod. year'])

In [225]:
data

,Levy,Manufacturer,Model,Category,Leather interior,Fuel type,Engine volume,Mileage,Gear box type,Drive wheels,Wheel,Color,Airbags,Age
0,831,HYUNDAI,Sonata,Sedan,Yes,Petrol,2.4,161600,Tiptronic,Front,Left wheel,Red,8,15


In [226]:
## One-hot Encoding Categorical Columns
one_hot_encoding_cols = ["Leather interior", "Gear box type", "Drive wheels", "Wheel"]

## Load the saved one-hot encoders
with open("../models/one_hot_encoder.pkl", "rb") as f:
    one_hot_encoded_data = pickle.load(f)

## One-hot encoding for the new data
encoded_data = one_hot_encoded_data.transform(data[one_hot_encoding_cols])
encoded_columns = one_hot_encoded_data.get_feature_names_out(one_hot_encoding_cols)
encoded_data_df = pd.DataFrame(encoded_data, columns=encoded_columns, index=data.index)

## Concatenate the one-hot encoded data and drop original columns
data = pd.concat([data, encoded_data_df], axis=1)
data.drop(columns=one_hot_encoding_cols, inplace=True)

In [227]:
## Label Encode Categorical Columns
label_encoding_cols = ["Manufacturer", "Model", "Category", "Color", "Fuel type"]

## Load the saved label encoders
with open('../models/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

## Apply label encoding
for column in label_encoding_cols:
    le = label_encoders[column]
    #! For Unseen Categorical
    data[column] = data[column].apply(lambda x: x if x in le.classes_ else le.classes_[0])
    data[column] = le.transform(data[column])

In [228]:
data

,Levy,Manufacturer,Model,Category,Fuel type,Engine volume,Mileage,Color,Airbags,Age,...,Leather interior_Yes,Gear box type_Automatic,Gear box type_Manual,Gear box type_Tiptronic,Gear box type_Variator,Drive wheels_4x4,Drive wheels_Front,Drive wheels_Rear,Wheel_Left wheel,Wheel_Right-hand drive
0,831,16,578,8,4,2.4,161600,11,8,15,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [229]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Levy                     1 non-null      int64  
 1   Manufacturer             1 non-null      int64  
 2   Model                    1 non-null      int64  
 3   Category                 1 non-null      int64  
 4   Fuel type                1 non-null      int64  
 5   Engine volume            1 non-null      float64
 6   Mileage                  1 non-null      int64  
 7   Color                    1 non-null      int64  
 8   Airbags                  1 non-null      int64  
 9   Age                      1 non-null      int64  
 10  Leather interior_No      1 non-null      float64
 11  Leather interior_Yes     1 non-null      float64
 12  Gear box type_Automatic  1 non-null      float64
 13  Gear box type_Manual     1 non-null      float64
 14  Gear box type_Tiptronic  1 non-null      

In [230]:
## Scale column
with open("../models/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
data = scaler.transform(data)

In [231]:
with open("../models/LRModel.pkl", "rb") as f:
    LRModel = pickle.load(f)

print(LRModel.predict(data))
print(f"The Price is = {LRModel.predict(data)[0]}")

[16844.38182899]
The Price is = 16844.381828988226


In [232]:
with open('../models/RFModel.pkl', 'rb') as f:
    RFModel = pickle.load(f)
    
print(RFModel.predict(data))
print(f"The Price is = {RFModel.predict(data)[0]}")   

[15516.93]
The Price is = 15516.93
